# Reproduce Paper v7 Section 11 Synthetic Results

> This notebook generates synthetic toy covariance matrices only. It does not use real CMB data, Planck maps, CLASS/CAMB, COMPACT eigenmodes, masks, beams, foreground models, or a Planck likelihood. It is a proof-of-method companion for reproducing the synthetic Section 11 outputs in the paper draft.

Run the cells from top to bottom after installing the package with `python -m pip install -e ".[dev]"`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from viscollapse.figures import plot_covariance_blocks, plot_injected_recovered, plot_sn_scan
from viscollapse.paper import (
    build_paper_reproduction,
    validate_paper_reproduction,
    write_paper_outputs,
)

results_dir = Path("../results/paper_v7").resolve()
figures_dir = Path("../figures/paper_v7").resolve()
bundle = build_paper_reproduction()
failures = validate_paper_reproduction(bundle)
failures

## 1. Projection Check

The visible-sector odd projection vanishes synthetically: `P_vis tau_1 P_vis = 0`. The even identity projection remains nonzero.

In [ ]:
pd.DataFrame([bundle.projection])

## 2. 6x6 Covariance Summary

The 6x6 matrix is a toy off-diagonal covariance block used to check algebra and estimator normalization. It is not a physical covariance prediction.

In [ ]:
bundle.covariance_summary.query("template == '6x6 toy block'")

## 3. 117-mode Synthetic Covariance Summary

The 117-mode template uses the mode count for `2 <= ell <= 10` and is normalized so `F_A = 1`.

In [ ]:
bundle.covariance_summary.query("template == '117-mode synthetic ring'")

## 4. Bias-subtracted Quadratic Estimator

For synthetic Gaussian samples with covariance `C = I + A_inj Delta C`, the toy estimator is

$$\hat A = \frac{x^T \Delta C x - \mathrm{Tr}(\Delta C)}{\mathrm{Tr}(\Delta C^2)}.$$

This estimator sanity check assumes `C0 = I` and synthetic off-diagonal templates only.

## 5. 6x6 Monte Carlo Recovery Table

In [ ]:
bundle.mc6[["A_inj", "Ahat_mean", "Ahat_SE", "status"]]

## 6. 117-mode Monte Carlo Recovery Table

In [ ]:
bundle.mc117[["A_inj", "Ahat_mean", "Ahat_SE", "status"]]

## 7. lambda_b--f_phi_paired Scan

In [ ]:
bundle.scan.round(4)

## 8. Write Tables and Figures

The same output paths are used by `python scripts/run_prototype.py`.

In [ ]:
write_paper_outputs(bundle, results_dir)
figures_dir.mkdir(parents=True, exist_ok=True)
plot_covariance_blocks(figures_dir / "covariance_blocks.png")
plot_sn_scan(bundle.scan, figures_dir / "sn_scan_lambda_fpaired.png")
plot_injected_recovered(bundle.mc117, figures_dir / "injected_recovered.png")
sorted(path.name for path in results_dir.glob("*")), sorted(path.name for path in figures_dir.glob("*"))

## 9. Generated Figures

In [ ]:
for name in ["covariance_blocks.png", "sn_scan_lambda_fpaired.png", "injected_recovered.png"]:
    display(Image(filename=str(figures_dir / name)))

## What This Does and Does Not Prove

This notebook reproduces the synthetic Section 11 proof-of-method outputs: projection checks, toy covariance summaries, deterministic estimator recovery, analytic scan tables, and synthetic figures.

It does not use real CMB data, does not run a Planck likelihood, does not run CLASS/CAMB, does not compute COMPACT eigenmodes, and does not claim observational validation.